# Filtered FCS: Basic Filter Computation

This notebook demonstrates the basic workflow for computing lifetime filters for filtered FCS (fFCS/FLCS) analysis.

## What are lifetime filters?

Lifetime filters allow you to separate signals from different fluorescent species based on their fluorescence lifetime signatures. In FCS experiments, you can weight each photon by its filter value to isolate the dynamics of individual species.

## Algorithm

The filter computation follows PAM's `Calc_fFCS_Filters` implementation:

1. Normalize species decay patterns
2. Create weight matrix W = diag(1/total_decay)
3. Compute filter matrix: F = (D^T W D)^(-1) D^T W
4. Calculate reconstruction and residuals for quality assessment

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Import the FCS filter calculator API
import sys
sys.path.insert(0, str(Path.cwd().parent.parent.parent.parent))
from chisurf.plugins.fcs.fcs_filter_calculator import compute_filters, FilterResult

## Step 1: Generate Synthetic Test Data

We'll create two species with different lifetimes and a mixed total decay.

In [ ]:
# Parameters
n_bins = 256
time = np.linspace(0, 20, n_bins)  # nanoseconds

# Species 1: Short lifetime (1 ns)
tau1 = 1.0
decay1 = np.exp(-time / tau1)
decay1 = decay1 / decay1.sum() * 10000  # Scale to realistic counts

# Species 2: Long lifetime (3 ns)
tau2 = 3.0
decay2 = np.exp(-time / tau2)
decay2 = decay2 / decay2.sum() * 10000

# Total decay: 60% species 1, 40% species 2
w1, w2 = 0.6, 0.4
total = w1 * decay1 + w2 * decay2

# Add Poisson noise
rng = np.random.RandomState(42)
total_noisy = rng.poisson(total)
decay1_noisy = rng.poisson(decay1)
decay2_noisy = rng.poisson(decay2)

print(f"Created test data:")
print(f"  TAC bins: {n_bins}")
print(f"  Species 1: τ = {tau1} ns, weight = {w1*100:.0f}%")
print(f"  Species 2: τ = {tau2} ns, weight = {w2*100:.0f}%")
print(f"  Total photons: {total_noisy.sum():.0f}")

## Step 2: Visualize Input Decay Patterns

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Linear scale
ax1.plot(time, total_noisy, 'k-', label='Total decay', lw=2)
ax1.plot(time, decay1_noisy, 'b--', label=f'Species 1 (τ={tau1} ns)', alpha=0.7)
ax1.plot(time, decay2_noisy, 'r--', label=f'Species 2 (τ={tau2} ns)', alpha=0.7)
ax1.set_xlabel('Time (ns)')
ax1.set_ylabel('Counts')
ax1.set_title('Decay Patterns (Linear)')
ax1.legend()
ax1.grid(alpha=0.3)

# Log scale
ax2.semilogy(time, total_noisy, 'k-', label='Total decay', lw=2)
ax2.semilogy(time, decay1_noisy, 'b--', label=f'Species 1', alpha=0.7)
ax2.semilogy(time, decay2_noisy, 'r--', label=f'Species 2', alpha=0.7)
ax2.set_xlabel('Time (ns)')
ax2.set_ylabel('Counts (log)')
ax2.set_title('Decay Patterns (Log)')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Step 3: Compute Lifetime Filters

Use the API to compute filters from the decay patterns.

In [ ]:
# Compute filters
result = compute_filters(
    total_decay=total_noisy,
    species_decays=[decay1_noisy, decay2_noisy],
    metadata={"tau1_ns": tau1, "tau2_ns": tau2, "w1": w1, "w2": w2}
)

print(f"\nFilter computation results:")
print(f"  Filter shape: {result.filters.shape}")
print(f"  Species: {result.n_species}")
print(f"  TAC bins: {result.n_bins}")
print(f"  Max weighted residual: {np.abs(result.weighted_residuals).max():.3f}")

## Step 4: Visualize Computed Filters

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

# Filter shapes
ax1.plot(time, result.filters[0, :], 'b-', label='Filter 1 (short lifetime)', lw=2)
ax1.plot(time, result.filters[1, :], 'r-', label='Filter 2 (long lifetime)', lw=2)
ax1.axhline(0, color='k', linestyle='--', alpha=0.3)
ax1.set_xlabel('Time (ns)')
ax1.set_ylabel('Filter value')
ax1.set_title('Lifetime Filters')
ax1.legend()
ax1.grid(alpha=0.3)

# Reconstruction quality
ax2.semilogy(time, total_noisy, 'ko', markersize=3, alpha=0.5, label='Total decay (data)')
ax2.semilogy(time, result.reconstruction, 'r-', lw=2, label='Reconstruction')
ax2.set_xlabel('Time (ns)')
ax2.set_ylabel('Counts (log)')
ax2.set_title('Reconstruction Quality')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Step 5: Assess Quality with Weighted Residuals

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 4))

ax.plot(time, result.weighted_residuals, 'g-', lw=1)
ax.axhline(0, color='k', linestyle='--', alpha=0.5)
ax.axhline(3, color='r', linestyle=':', alpha=0.5, label='±3σ')
ax.axhline(-3, color='r', linestyle=':', alpha=0.5)
ax.set_xlabel('Time (ns)')
ax.set_ylabel('Weighted residuals')
ax.set_title('Weighted Residuals: (Total - Reconstruction) / sqrt(Total)')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Residual statistics:")
print(f"  Mean: {result.weighted_residuals.mean():.3f}")
print(f"  Std: {result.weighted_residuals.std():.3f}")
print(f"  Range: [{result.weighted_residuals.min():.3f}, {result.weighted_residuals.max():.3f}]")

## Step 6: Export Filters to JSON

Save the computed filters for use in FCS correlation analysis.

In [ ]:
# Export to JSON
output_path = "fcs_filter.json"
result.to_json(output_path, indent=2)

print(f"✓ Filters exported to: {output_path}")

# Verify by loading back
loaded = FilterResult.from_json(output_path)
print(f"\n✓ Verified: loaded {loaded.n_species} species, {loaded.n_bins} bins")

## Next Steps

1. **Apply filters to photon data**: Use the filter values to weight photons in your FCS correlation
2. **Species-specific correlations**: Compute separate correlation curves for each species
3. **Check the tttrlib example**: See how to apply these filters to real TTTR data

## Filter Interpretation

- **Positive filter values** at early times → emphasize short-lifetime species
- **Positive filter values** at late times → emphasize long-lifetime species
- **Filter sum**: Each filter integrates the contribution of its target species
- **Good fit**: Weighted residuals should be randomly distributed around zero with σ ≈ 1